[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/04_research.ipynb)

# 04 · Research Agent — AI-powered analysis

The **[Bigdata.com](https://bigdata.com)** Research Agent (`POST /v1/research-agent`)
goes beyond retrieval: it **plans** a research approach, runs multiple searches,
and **synthesises a cited answer** grounded in the underlying documents.

**What it's good for**
- One-shot answers to analytical questions ("What are the key risks for X?")
- Multi-step research that would otherwise need many manual searches
- Grounded, source-referenced narratives you can drop into a report

**Important:** this endpoint **streams** its progress as Server-Sent Events
(SSE) and lives on the **`agents.bigdata.com`** host. Each `data:` line is a JSON
event; the message `type` tells you what it is (`THINKING`, `PLANNING`,
`ACTION`, `ANSWER`, `COMPLETE`, ...). We collect the `ANSWER` chunks for the
final response.

In [ ]:
!pip install -q requests env-colab-pass

In [ ]:
import requests
from env_colab_pass import passutil
from IPython.display import Markdown, display

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = passutil.get_secret_value("BIGDATA_API_KEY")

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

### A helper to consume the SSE stream
It yields parsed events and concatenates the `ANSWER` text into the final answer.

In [ ]:
import json

def run_research(message, research_effort="lite", tools_configs=None, show_progress=True):
    """Stream the Research Agent and return the final answer text.
    research_effort: 'lite' (quick) or 'standard' (thorough)."""
    payload = {"message": message, "research_effort": research_effort}
    if tools_configs:
        payload["tools_configs"] = tools_configs

    answer_parts, plan_printed = [], set()
    with requests.post(f"{AGENTS_BASE}/v1/research-agent", headers=HEADERS, json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[len("data:"):].strip())
            msg = event.get("message", {})
            mtype = msg.get("type")
            if mtype == "PLANNING" and show_progress:
                for step in msg.get("plan", {}).get("steps", []):
                    if step["description"] not in plan_printed:
                        plan_printed.add(step["description"])
                        print("  plan:", step["description"])
            elif mtype == "ACTION" and show_progress:
                print("  action:", msg.get("tool_name"))
            elif mtype == "ANSWER":
                answer_parts.append(msg.get("content", ""))
    return "".join(answer_parts)

### Example 1 — A quick (lite) research question

In [ ]:
answer = run_research("What are the main risks Apple highlighted recently?", research_effort="lite")
print("\n=== ANSWER ===\n")


In [ ]:
display(Markdown(answer))

### Example 2 — Constrain the research with filters
`tools_configs.search.query_filters` lets you scope the agent's searches — here
to a specific entity and time period. Use `ranking_parameters` to bias freshness
or source quality (1-10).

In [ ]:
response = requests.post(
    f"{API_BASE}/v1/knowledge-graph/companies/isin",
    headers=HEADERS,
    json={"values": ["US0378331005"]},
    timeout=30
)
apple = response.json()["results"]["US0378331005"]["id"]

print(apple)

answer = run_research(
    "Summarise the most recent demand trends discussed.",
    research_effort="lite",
    tools_configs={
        "search": {
            "query_filters": {
                "entities": {"any_of": [apple]},
                "period": {"start": "2025-01-01T00:00:00Z", "end": "2025-12-31T23:59:59Z"},
            },
            "ranking_parameters": {"freshness_boost": 5},
        }
    },
)



In [ ]:
print("\n=== ANSWER ===\n")
display(Markdown(answer))

## Notes
- **`research_effort`**: `lite` for fast answers, `standard` for deeper analysis.
- **`model_name`**: `base` (default) or `pro` for harder reasoning tasks.
- **Follow-ups**: set `persistence_mode="enabled"` and reuse the returned
  `chat_id` to continue a conversation.
- **Citation Details**: Refer [Cookbook on github](https://github.com/Bigdata-com/bigdata-cookbook/blob/main/Research_Agent_Sync_Response/research_client_usage.ipynb) 
- **Structured output**: pass `structured_output_schema` (a JSON Schema) to get
  machine-readable extraction instead of prose.

**Next:** [`05_workflows.ipynb`](05_workflows.ipynb)

_Source: [Bigdata.com](https://bigdata.com) · Research docs: https://docs.bigdata.com/api-reference/research-agent/research-agent_